# Improving Predictive Accuracy for Agile Software Effort Estimation using Advanced Optimization and Hybrid Techniques

Experiment notebook for the AgES dataset (Raw_AgES_Dataset.csv). One run produces every number, table and figure used in the manuscript.

**How to use**
1. Upload `Raw_AgES_Dataset.csv` to the Colab session (left panel, Files) or mount Drive and change the path in the first code cell (`AGES_DATA`).
2. Run the install cell, then run all cells. Start with `PROFILE = "fast"` (roughly one hour, an estimate rather than a measurement) to confirm everything works, then set `PROFILE = "full"` (several hours; the nested tuning dominates the run time).
3. Results are cached in `ages_outputs/`. If the session disconnects, run all cells again; finished experiments are reloaded, not recomputed.
4. The final cell packs everything into `ages_outputs.zip`.

**Design in one paragraph.** The target is log(1 + days). Three evaluation protocols are used throughout: (R) repeated random 5-fold, (G) repository-grouped 5-fold, so that every test repository is unseen, and (T) a chronological split, training only on issues closed before the test period starts. Features are grouped by when their value becomes known. All tuning is nested: hyperparameters, feature selection and the stacking meta-learner are fitted on training data only, and the validation folds used inside the training data follow the same rule as the outer protocol (random, repository-grouped or time-blocked), so that tuning is aimed at the kind of generalisation that is being measured.

In [ ]:
import os
# Choose "fast" for a first check and "full" for the final run.
os.environ["AGES_PROFILE"] = "fast"
# Place Raw_AgES_Dataset.csv in the session storage (Files panel), or mount Drive and change the path.
os.environ["AGES_DATA"] = "Raw_AgES_Dataset.csv"
os.environ["AGES_OUT"] = "ages_outputs"
assert os.path.exists(os.environ["AGES_DATA"]), "Upload Raw_AgES_Dataset.csv first"


In [ ]:
!pip install -q optuna lightgbm catboost xgboost
import os, sys, json, time, warnings, ast, pickle, platform, zipfile
import numpy as np, pandas as pd
from scipy import stats
import sklearn
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, r2_score
warnings.filterwarnings("ignore")

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
try:
    import lightgbm as lgb
except Exception:
    lgb = None
try:
    import catboost as cb
except Exception:
    cb = None

# ------------------------------------------------------------------ configuration
PROFILE   = os.environ.get("AGES_PROFILE", "fast")        # "fast" or "full"
DATA_PATH = os.environ.get("AGES_DATA", "Raw_AgES_Dataset.csv")
OUT       = os.environ.get("AGES_OUT", "ages_outputs")
DRY_ROWS  = int(os.environ.get("AGES_DRY_ROWS", "0"))      # >0 only for smoke tests
SEED      = 42
PROFILES = {
    "fast": dict(repeats=1, n_trials=8,  rf_trials=3, hgb_trials=4,  strategy_cmp=False, perm_repeats=2),
    "full": dict(repeats=2, n_trials=25, rf_trials=8, hgb_trials=10, strategy_cmp=True,  perm_repeats=5),
}
CFG = PROFILES[PROFILE]
VT_THRESHOLD = 0.0475      # keeps binary columns with prevalence above about 5 percent (p(1-p) > 0.0475)
os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)

ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
           scipy=__import__("scipy").__version__, sklearn=sklearn.__version__, xgboost=xgb.__version__,
           optuna=optuna.__version__, lightgbm=getattr(lgb, "__version__", None), catboost=getattr(cb, "__version__", None),
           profile=PROFILE, config=CFG, seed=SEED, vt_threshold=VT_THRESHOLD)
print(json.dumps(ENV, indent=1))

## 1. Data and feature construction

`Raw_AgES_Dataset.csv` holds 23,006 closed issues with 15 attributes. Every model feature below is rebuilt from these columns, so no private intermediate file is needed. The target is `log1p(Time Taken (days))`; `Time Taken (days)` is the whole number of days between `Created At` and `Closed At`.

Feature groups, by the moment the value is known:

| Group | Features | Known when |
|---|---|---|
| Post-hoc | `Comments` | accumulates until closure |
| Triage | `AssigneeCount`, `HasMilestone`, `ExpertiseRelease` (static score in the release), `Category_*`, `Component_*`, `n_labels` | after the issue is labelled and assigned |
| Creation-time (engineered) | title and body length, code and link flags, calendar fields, prior history of the repository and of the assignee, current open workload | at creation, using only issues closed before that moment |

Two columns of the earlier released `cleaned_dataset.csv` are not features: `Issue Age (days)` is an exact copy of the scaled target, and `Complexity` is a rescaled copy of `Comments` (identical for most rows). Both are excluded.

In [ ]:
raw = pd.read_csv(DATA_PATH)
if DRY_ROWS:
    raw = raw.sample(DRY_ROWS, random_state=SEED).reset_index(drop=True)
raw["created"] = pd.to_datetime(raw["Created At"], utc=True)
raw["closed"]  = pd.to_datetime(raw["Closed At"],  utc=True)
EPOCH = pd.Timestamp("1970-01-01", tz="UTC")
raw["t_created"] = (raw["created"] - EPOCH).dt.total_seconds().values
raw["t_closed"]  = (raw["closed"]  - EPOCH).dt.total_seconds().values
days = raw["Time Taken (days)"].astype(float).values
y = np.log1p(days)
YMAX = float(y.max())
repo = raw["Repository"].values
n = len(raw)
assert (raw["t_closed"] >= raw["t_created"]).all()
assert (np.floor((raw["t_closed"] - raw["t_created"]) / 86400) == days).mean() > 0.999

DATA_FACTS = dict(n_issues=int(n), n_repositories=int(pd.Series(repo).nunique()),
                  created_min=str(raw.created.min()), created_max=str(raw.created.max()),
                  days_mean=float(days.mean()), days_median=float(np.median(days)), days_max=float(days.max()),
                  share_zero_day=float((days == 0).mean()), log_target_median=float(np.median(y)),
                  log_target_var=float(y.var()))
END_CLOSED = raw["t_closed"].max()
MATURE = (raw["t_created"].values <= END_CLOSED - 365 * 86400)   # duration observable for at least one year
DATA_FACTS.update(share_created_2024=float((raw.created.dt.year == 2024).mean()), n_mature=int(MATURE.sum()), last_closure=str(raw.closed.max()))
print(DATA_FACTS)
SAMPLING = raw.groupby("Repository").agg(n=("created", "size"), first_created=("created", "min"), last_created=("created", "max"), median_days=("Time Taken (days)", "median")).sort_values("n", ascending=False)
SAMPLING.to_csv(os.path.join(OUT, "table_repository_sampling_window.csv")); print(SAMPLING.to_string())

In [ ]:
def parse_list(s):
    if isinstance(s, str):
        try:
            v = ast.literal_eval(s)
            return list(v) if isinstance(v, (list, tuple)) else [str(v)]
        except Exception:
            return []
    return []

assignees = raw["Assignees"].apply(parse_list)
primary = assignees.apply(lambda a: a[0] if len(a) else "NA").values
labels = raw["Labels"].apply(parse_list)
body = raw["Body"].fillna("").astype(str)
title = raw["Title"].fillna("").astype(str)

def prior_stats(keys, tc, tz, yy):
    """For each issue: number and sum of target values of issues of the same key that were closed strictly before this issue was created."""
    n_prior = np.zeros(len(yy)); s_prior = np.zeros(len(yy)); n_open = np.zeros(len(yy))
    df = pd.DataFrame({"k": keys, "c": tc, "z": tz, "y": yy, "i": np.arange(len(yy))})
    for _, g in df.groupby("k", sort=False):
        order = np.argsort(g["z"].values, kind="stable")
        zs = g["z"].values[order]; cs = np.concatenate([[0.0], np.cumsum(g["y"].values[order])])
        idx = np.searchsorted(zs, g["c"].values, side="left")
        n_prior[g["i"].values] = idx; s_prior[g["i"].values] = cs[idx]
        cr = np.sort(g["c"].values)
        opened = np.searchsorted(cr, g["c"].values, side="right")
        closed = np.searchsorted(zs, g["c"].values, side="right")
        n_open[g["i"].values] = np.clip(opened - closed - 1, 0, None)
    return n_prior, s_prior, n_open

# global expanding mean (all issues closed before creation) used as the shrinkage target
gn, gs, _ = prior_stats(np.zeros(n), raw["t_created"].values, raw["t_closed"].values, y)
gmean = np.where(gn > 0, gs / np.maximum(gn, 1), y.mean())
M_SHRINK = 5.0
def shrunk(nn, ss): return (ss + M_SHRINK * gmean) / (nn + M_SHRINK)

rn, rs, ro = prior_stats(repo, raw["t_created"].values, raw["t_closed"].values, y)
an, as_, ao = prior_stats(pd.Series(repo).astype(str).values + "|" + primary, raw["t_created"].values, raw["t_closed"].values, y)

F = pd.DataFrame(index=raw.index)
# release features
F["Comments"] = raw["Comments"].astype(float)
F["AssigneeCount"] = assignees.apply(len).astype(float)
F["HasMilestone"] = raw["Milestone"].notna().astype(float)
F["ExpertiseRelease"] = raw["Contributors Expertise"].astype(float)
F["n_labels"] = labels.apply(len).astype(float)
cat = pd.get_dummies(raw["Category"].fillna("Unknown"), prefix="Category").astype(float)
comp = pd.get_dummies(raw["Component"].fillna("Unknown"), prefix="Component").astype(float)
CAT_COLS, COMP_COLS = list(cat.columns), list(comp.columns)
F = pd.concat([F, cat, comp], axis=1)
# engineered creation-time features
F["title_len"] = title.str.len().astype(float)
F["title_words"] = title.str.split().apply(len).astype(float)
F["body_len_log"] = np.log1p(body.str.len().astype(float))
F["body_has_code"] = body.str.contains("```", regex=False).astype(float)
F["body_n_links"] = body.str.count("http").astype(float)
F["created_year"] = raw["created"].dt.year.astype(float) - 2011
F["created_dow"] = raw["created"].dt.dayofweek.astype(float)
F["created_hour"] = raw["created"].dt.hour.astype(float)
F["repo_prior_n_log"] = np.log1p(rn); F["repo_prior_mean"] = shrunk(rn, rs); F["repo_open_log"] = np.log1p(ro)
F["asg_prior_n_log"] = np.log1p(an); F["asg_prior_mean"] = shrunk(an, as_); F["asg_open_log"] = np.log1p(ao)
ENG_COLS = ["title_len", "title_words", "body_len_log", "body_has_code", "body_n_links", "created_dow",
            "created_hour", "repo_prior_n_log", "repo_prior_mean", "repo_open_log", "asg_prior_n_log", "asg_prior_mean", "asg_open_log"]

FEATURE_SETS = {
    "S0": ["Comments", "AssigneeCount", "HasMilestone", "ExpertiseRelease"] + CAT_COLS + COMP_COLS,      # the feature set of the earlier release pipeline
    "S1": ["AssigneeCount", "HasMilestone", "ExpertiseRelease"] + CAT_COLS + COMP_COLS,                  # S0 without Comments
    "S2": ["ExpertiseRelease"] + CAT_COLS + COMP_COLS,                                                  # the attribute set retained by the base paper
    "S3": CAT_COLS + COMP_COLS + ["n_labels"] + ENG_COLS,                                               # leakage-free planning set (primary)
    "S3b": CAT_COLS + COMP_COLS + ["n_labels", "AssigneeCount", "HasMilestone"] + ENG_COLS,              # S3 plus triage fields
    "S3c": CAT_COLS + COMP_COLS + ["n_labels"] + [c for c in ENG_COLS if not c.startswith("repo_")],                  # S3 without repository-level history (diagnostic ablation)
    "S4": ["Comments", "AssigneeCount", "HasMilestone", "ExpertiseRelease", "n_labels", "created_year"] + CAT_COLS + COMP_COLS + ENG_COLS,  # everything
}
XS = {k: F[v].values.astype(np.float32) for k, v in FEATURE_SETS.items()}
print({k: v.shape[1] for k, v in XS.items()})
# how static is the released expertise score?
chk = pd.DataFrame({"r": repo, "a": raw["Assignees"].astype(str).values, "e": raw["Contributors Expertise"].values}).groupby(["r", "a"])["e"].nunique()
DATA_FACTS["expertise_distinct_values_per_repo_assignee_max"] = int(chk.max())
DATA_FACTS["spearman_comments_vs_target"] = float(stats.spearmanr(F["Comments"], y)[0])
DATA_FACTS["spearman_expertise_release_vs_target"] = float(stats.spearmanr(F["ExpertiseRelease"], y)[0])
DATA_FACTS["spearman_repo_prior_mean_vs_target"] = float(stats.spearmanr(F["repo_prior_mean"], y)[0])
DATA_FACTS["spearman_asg_prior_mean_vs_target"] = float(stats.spearmanr(F["asg_prior_mean"], y)[0])
print({k: v for k, v in DATA_FACTS.items() if k.startswith(("spearman", "expertise"))})

## 2. Evaluation protocols and metrics

In [ ]:
def group_folds(repos, k, seed):
    rng = np.random.RandomState(seed)
    sizes = pd.Series(repos).value_counts()
    order = sorted(sizes.index, key=lambda r: -sizes[r] * (1 + 0.35 * rng.rand()))
    load = np.zeros(k); assign = {}
    for r in order:
        f = int(np.argmin(load + 1e-6 * rng.rand(k))); assign[r] = f; load[f] += sizes[r]
    fid = np.array([assign[r] for r in repos])
    return [(np.where(fid != f)[0], np.where(fid == f)[0]) for f in range(k)]

FOLDS = {"R": [], "G": [], "T": []}
for rep in range(CFG["repeats"]):
    for f, (tr, te) in enumerate(KFold(5, shuffle=True, random_state=SEED + rep).split(np.arange(n))):
        FOLDS["R"].append((rep, f, tr, te))
    for f, (tr, te) in enumerate(group_folds(repo, 5, SEED + rep)):
        FOLDS["G"].append((rep, f, tr, te))
# Chronological protocol. The release contains only closed issues, so issues created close to the end of collection
# are biased towards short durations (right censoring). The test window is therefore the twelve months that end one
# year before the last recorded closure; training uses only issues closed before the window opens.
win_end = END_CLOSED - 365 * 86400
cut = win_end - 365 * 86400
te_T = np.where((raw["t_created"].values >= cut) & (raw["t_created"].values < win_end))[0]
tr_T = np.where(raw["t_closed"].values < cut)[0]
FOLDS["T"].append((0, 0, tr_T, te_T))
PROTO_INFO = {p: dict(n_folds=len(v), mean_train=float(np.mean([len(a[2]) for a in v])), mean_test=float(np.mean([len(a[3]) for a in v]))) for p, v in FOLDS.items()}
PROTO_INFO["T"]["window_start"] = str(pd.Timestamp(cut, unit="s", tz="UTC")); PROTO_INFO["T"]["window_end"] = str(pd.Timestamp(win_end, unit="s", tz="UTC"))
print(PROTO_INFO)

def metrics(yt, yp, dt):
    e = yp - yt
    pd_ = np.clip(np.expm1(yp), 0, None)
    return dict(MAE=float(np.mean(np.abs(e))), MdAE=float(np.median(np.abs(e))), MSE=float(np.mean(e ** 2)),
                RMSE=float(np.sqrt(np.mean(e ** 2))), R2=float(r2_score(yt, yp)),
                MAE_days=float(np.mean(np.abs(pd_ - dt))), MdAE_days=float(np.median(np.abs(pd_ - dt))))

STORE = {}
def store_path(key): return os.path.join(OUT, "cache_" + key.replace("|", "__").replace("/", "_") + ".pkl")
def save_store(key):
    with open(store_path(key), "wb") as fh: pickle.dump(STORE[key], fh)
def load_store(key):
    p = store_path(key)
    if os.path.exists(p):
        with open(p, "rb") as fh: STORE[key] = pickle.load(fh)
        return True
    return False

def new_entry(fs): return dict(fs=fs, rows=[], oof={p: {} for p in FOLDS}, extra=[])

def record(key, proto, rep, f, te, pred, ntr):
    ent = STORE[key]
    pred = np.clip(pred, 0.0, YMAX)      # the target lies in [0, log1p(max days)]; clipping removes numerical blow-ups of linear models
    m = metrics(y[te], pred, days[te]); m.update(proto=proto, rep=rep, fold=f, n_train=int(ntr), n_test=int(len(te)))
    ent["rows"].append(m)
    arr = ent["oof"][proto].setdefault(rep, np.full(n, np.nan)); arr[te] = pred

def run_model(key, fs, build, protos=("R", "G", "T"), reps=None, mask=None):
    if load_store(key) and all(p in STORE[key]["oof"] and STORE[key]["oof"][p] for p in protos): return
    STORE[key] = new_entry(fs); t0 = time.time()
    for p in protos:
        for rep, f, tr, te in FOLDS[p]:
            if reps is not None and rep >= reps: continue
            if mask is not None: tr, te = tr[mask[tr]], te[mask[te]]
            m = build(); m.fit(XS[fs][tr], y[tr]); record(key, p, rep, f, te, m.predict(XS[fs][te]), len(tr))
    save_store(key); print(f"[{key}] {time.time()-t0:.0f}s")

def run_multi(prefix, fs, fn, names, protos=("R", "G", "T"), reps=None):
    keys = [f"{prefix}_{nm}|{fs}" for nm in names]
    if all(load_store(k) for k in keys) and all(all(STORE[k]["oof"][p] for p in protos) for k in keys): return
    for k in keys: STORE[k] = new_entry(fs)
    t0 = time.time()
    for p in protos:
        for rep, f, tr, te in FOLDS[p]:
            if reps is not None and rep >= reps: continue
            preds, extra = fn(XS[fs][tr], y[tr], XS[fs][te], SEED + 100 * rep + f, dict(repo=repo[tr], t=raw["t_created"].values[tr], proto=p))
            for nm, k in zip(names, keys):
                record(k, p, rep, f, te, preds[nm], len(tr))
            extra.update(proto=p, rep=rep, fold=f); STORE[keys[0]]["extra"].append(extra)
    for k in keys: save_store(k)
    print(f"[{prefix}|{fs}] {time.time()-t0:.0f}s")

def summarize(key, proto):
    r = pd.DataFrame(STORE[key]["rows"]); r = r[r.proto == proto]
    cols = ["MAE", "MdAE", "MSE", "RMSE", "R2", "MAE_days", "MdAE_days"]
    out = r[cols].mean().to_dict(); out.update({c + "_sd": float(r[c].std(ddof=1)) if len(r) > 1 else np.nan for c in cols}); out["n_folds"] = int(len(r))
    return out

## 3. Models and nested tuning

In [ ]:
def make_xgb(**kw):
    base = dict(tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0)
    base.update(kw); return xgb.XGBRegressor(**base)

XGB_REF = dict(n_estimators=150, max_depth=6, learning_rate=0.05)       # reference configuration (other parameters at library defaults)
def b_xgb_default(): return make_xgb()
def b_xgb_ref():     return make_xgb(**XGB_REF)
def b_xgb_ref_vt():  return Pipeline([("vt", VarianceThreshold(VT_THRESHOLD)), ("m", make_xgb(**XGB_REF))])
def b_const():       return DummyRegressor(strategy="median")
def b_ridge():       return make_pipeline(StandardScaler(), Ridge(alpha=10.0))
def b_rf():          return RandomForestRegressor(n_estimators=150, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=SEED)
def b_hgb():         return HistGradientBoostingRegressor(random_state=SEED)
def b_lgbm():        return lgb.LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1)
def b_cat():         return cb.CatBoostRegressor(random_seed=SEED, verbose=0, thread_count=-1)

def inner_folds(ctx, k, seed):
    """Validation folds inside a training set, matched to the outer protocol: random for R, repository-grouped for G, contiguous time blocks for T."""
    m = len(ctx["t"])
    if ctx["proto"] == "G":
        gf = group_folds(ctx["repo"], k, seed)
        return [(a, b) for a, b in gf]
    if ctx["proto"] == "T":
        order = np.argsort(ctx["t"], kind="stable"); blocks = np.array_split(order, k)
        return [(np.concatenate([blocks[j] for j in range(k) if j != i]), blocks[i]) for i in range(k)]
    return list(KFold(k, shuffle=True, random_state=seed).split(np.arange(m)))

def tune_xgb(X, yy, n_trials, sampler, seed, folds):
    def obj(trial):
        p = dict(n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
                 max_depth=trial.suggest_int("max_depth", 3, 10),
                 learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                 subsample=trial.suggest_float("subsample", 0.5, 1.0),
                 colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                 min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                 gamma=trial.suggest_float("gamma", 0.0, 0.5),
                 reg_lambda=trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
                 reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True))
        errs = []
        for a, b in folds:
            m = make_xgb(**p); m.fit(X[a], yy[a]); errs.append(np.mean(np.abs(m.predict(X[b]) - yy[b])))
        return float(np.mean(errs))
    sam = optuna.samplers.TPESampler(seed=seed) if sampler == "tpe" else optuna.samplers.RandomSampler(seed=seed)
    st = optuna.create_study(direction="minimize", sampler=sam); st.optimize(obj, n_trials=n_trials)
    return st.best_params, [float(v) for v in np.minimum.accumulate([t.value for t in st.trials])]

def random_search(make, sample, X, yy, n_trials, seed, folds):
    rng = np.random.RandomState(seed); best = (np.inf, None)
    for _ in range(n_trials):
        p = sample(rng); errs = []
        for a, b in folds:
            m = make(**p); m.fit(X[a], yy[a]); errs.append(np.mean(np.abs(m.predict(X[b]) - yy[b])))
        if np.mean(errs) < best[0]: best = (float(np.mean(errs)), p)
    return best[1]

def mk_rf(**p): return RandomForestRegressor(n_estimators=100, max_samples=0.7, n_jobs=-1, random_state=SEED, **p)
def sm_rf(r): return dict(min_samples_leaf=int(r.choice([1, 3, 5, 10, 20])), max_features=float(r.choice([0.3, 0.5, 0.8])), max_depth=r.choice([10, 16, 24]).item())
def mk_hgb(**p): return HistGradientBoostingRegressor(random_state=SEED, **p)
def sm_hgb(r): return dict(learning_rate=float(10 ** r.uniform(-2, -0.7)), max_leaf_nodes=int(r.choice([15, 31, 63, 127])), min_samples_leaf=int(r.choice([10, 20, 50, 100])), l2_regularization=float(10 ** r.uniform(-3, 1)), max_iter=int(r.choice([100, 200, 300])))

def stage_cd(Xtr, ytr, Xte, seed, ctx):
    """Stage C: VT selection plus tuned XGBoost (TPE). Stage D: Ridge stack of tuned XGBoost, tuned random forest and tuned histogram gradient boosting."""
    vt = VarianceThreshold(VT_THRESHOLD).fit(Xtr); A, B = vt.transform(Xtr), vt.transform(Xte)
    inner = inner_folds(ctx, 3, seed)
    pX, trace = tune_xgb(A, ytr, CFG["n_trials"], "tpe", seed, inner)
    pR = random_search(mk_rf, sm_rf, A, ytr, CFG["rf_trials"], seed, inner)
    pH = random_search(mk_hgb, sm_hgb, A, ytr, CFG["hgb_trials"], seed, inner)
    makers = [lambda: make_xgb(**pX), lambda: mk_rf(**pR), lambda: mk_hgb(**pH)]
    oof = np.zeros((len(A), 3))
    for a, b in inner:
        for j, mkr in enumerate(makers): oof[b, j] = mkr().fit(A[a], ytr[a]).predict(A[b])
    meta = RidgeCV(alphas=np.logspace(-3, 3, 13)).fit(oof, ytr)
    P = np.column_stack([mkr().fit(A, ytr).predict(B) for mkr in makers])
    return ({"C": P[:, 0], "D": meta.predict(P), "Rf": P[:, 1], "Hgb": P[:, 2]},
            dict(xgb=pX, rf=pR, hgb=pH, weights=[float(w) for w in meta.coef_], intercept=float(meta.intercept_),
                 alpha=float(meta.alpha_), n_selected=int(A.shape[1]), trace=trace))

## 4. Statistical comparison

* Protocol R (repeated random folds, overlapping training sets): the corrected resampled t-test of Nadeau and Bengio (2003), with the variance correction `1/J + n_test/n_train`.
* Protocol G (repository-grouped): the per-repository mean absolute error is the unit of analysis (independent clusters); Wilcoxon signed-rank test and a bootstrap interval over repositories.
* Protocol T (chronological): paired bootstrap over test issues.

Differences are always `error(reference) - error(candidate)`, so a positive value means the candidate is better.

In [ ]:
def abs_err(key, proto, rep):
    return np.abs(STORE[key]["oof"][proto][rep] - y)

def compare(ref, cand, proto, B=2000, seed=SEED):
    rng = np.random.RandomState(seed)
    if proto == "R":
        a = pd.DataFrame(STORE[ref]["rows"]); b = pd.DataFrame(STORE[cand]["rows"])
        a = a[a.proto == "R"].set_index(["rep", "fold"]); b = b[b.proto == "R"].set_index(["rep", "fold"])
        d = (a["MAE"] - b["MAE"]).values; J = len(d); ratio = (a["n_test"] / a["n_train"]).mean()
        m, v = d.mean(), d.var(ddof=1); se = np.sqrt((1 / J + ratio) * v) if v > 0 else np.nan
        t = m / se if se and se > 0 else np.nan; tc = stats.t.ppf(0.975, J - 1)
        return dict(proto=proto, mean_diff=float(m), rel_pct=float(100 * m / a["MAE"].mean()), ci=[float(m - tc * se), float(m + tc * se)],
                    p=float(2 * stats.t.sf(abs(t), J - 1)) if np.isfinite(t) else np.nan, test="corrected resampled t", units=int(J))
    if proto == "G":
        reps = sorted(STORE[ref]["oof"]["G"].keys())
        ea = np.nanmean([abs_err(ref, "G", r) for r in reps], axis=0); eb = np.nanmean([abs_err(cand, "G", r) for r in reps], axis=0)
        df = pd.DataFrame({"r": repo, "d": ea - eb, "a": ea}).groupby("r").agg(d=("d", "mean"), a=("a", "mean")).dropna()
        d = df["d"].values; boots = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(B)]
        try: p = float(stats.wilcoxon(d).pvalue)
        except Exception: p = np.nan
        return dict(proto=proto, mean_diff=float(d.mean()), rel_pct=float(100 * d.mean() / df["a"].mean()), ci=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
                    p=p, test="Wilcoxon over repositories", units=int(len(d)), repos_improved=int((d > 0).sum()))
    ea = abs_err(ref, "T", 0); eb = abs_err(cand, "T", 0); idx = ~np.isnan(ea); d = (ea - eb)[idx]
    boots = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(B)]
    return dict(proto=proto, mean_diff=float(d.mean()), rel_pct=float(100 * d.mean() / ea[idx].mean()), ci=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
                p=float(2 * min((np.array(boots) <= 0).mean(), (np.array(boots) >= 0).mean())), test="paired bootstrap over test issues", units=int(len(d)))

## 5. Experiments

**E1** Feature-availability ablation. **E2** Model comparison with library or reference settings. **E3** Feature selection (stage B). **E4** Nested hyperparameter optimisation (stage C) and search-strategy check. **E5** Hybrid stacking with equally tuned base learners (stage D). **E6** Day-scale analysis, triage ability and feature importance. **E7** Same-pipeline replication of the published benchmark.

In [ ]:
# E1 and E2: models without tuning
t_all = time.time()
for fs in ["S0", "S1", "S2", "S3", "S3b", "S3c", "S4"]:
    run_model(f"xgb_ref|{fs}", fs, b_xgb_ref)
for fs in ["S3", "S0"]:
    run_model(f"const|{fs}", fs, b_const)
    run_model(f"ridge|{fs}", fs, b_ridge)
    run_model(f"rf|{fs}", fs, b_rf)
    run_model(f"hgb|{fs}", fs, b_hgb)
    run_model(f"xgb_default|{fs}", fs, b_xgb_default)
    if lgb is not None: run_model(f"lgbm|{fs}", fs, b_lgbm)
    if cb is not None:  run_model(f"catboost|{fs}", fs, b_cat)

rows = []
for key in sorted(STORE):
    for p in FOLDS:
        s = summarize(key, p) if STORE[key]["oof"][p] else None
        if s is None: continue
        s.update(model=key.split("|")[0], feature_set=STORE[key]["fs"], protocol=p, subset=("mature" if key.endswith("|mature") else "all")); rows.append(s)
T_E12 = pd.DataFrame(rows)
T_E12.to_csv(os.path.join(OUT, "table_E1_E2_models_and_feature_sets.csv"), index=False)
print(T_E12[T_E12.protocol.isin(["R", "G", "T"])][["model", "feature_set", "protocol", "MAE", "RMSE", "R2", "MAE_days", "MdAE_days"]].round(4).to_string(index=False))

In [ ]:
# E3: feature selection (stage B) on the primary and the legacy feature set
for fs in ["S3", "S0"]:
    run_model(f"xgb_ref_vt|{fs}", fs, b_xgb_ref_vt)

# E4 and E5: stages C and D (nested). Primary set under all protocols, legacy set under R for comparison with the earlier release.
run_multi("cd", "S3", stage_cd, ["C", "D", "Rf", "Hgb"], protos=("R", "G", "T"))
run_multi("cd", "S0", stage_cd, ["C", "D", "Rf", "Hgb"], protos=("R",), reps=1)

In [ ]:
# E1b: sensitivity to right censoring. Only issues created at least one year before the last closure are used, so every duration up to one year is fully observable.
for fs in ["S0", "S1", "S2", "S3", "S4"]:
    run_model(f"xgb_ref|{fs}|mature", fs, b_xgb_ref, protos=("R", "G"), mask=MATURE)
run_model("const|S3|mature", "S3", b_const, protos=("R", "G"), mask=MATURE)
MATURE_TABLE = pd.DataFrame([dict(model=k.split("|")[0], feature_set=STORE[k]["fs"], protocol=p, **{m: v for m, v in summarize(k, p).items() if m in ("MAE", "RMSE", "R2", "MAE_days", "MdAE_days")})
                             for k in sorted(STORE) if k.endswith("|mature") for p in ("R", "G")])
MATURE_TABLE.to_csv(os.path.join(OUT, "table_sensitivity_mature_issues.csv"), index=False); print(MATURE_TABLE.round(4).to_string(index=False))
MATURE_CMP = [dict(ref="S0", cand="S3", **compare("xgb_ref|S0|mature", "xgb_ref|S3|mature", p)) for p in ("R", "G")]
print(pd.DataFrame(MATURE_CMP)[["ref", "cand", "proto", "mean_diff", "rel_pct", "p"]].round(4).to_string(index=False))

In [ ]:
# E4b: search strategy under the same trial budget (random search against TPE), primary set, protocol R, first repeat
STRATEGY = {}
if CFG["strategy_cmp"]:
    sp = os.path.join(OUT, "strategy.json")
    if os.path.exists(sp):
        STRATEGY = json.load(open(sp))
    else:
        for strat in ["random", "tpe"]:
            res = []
            for rep, f, tr, te in FOLDS["R"]:
                if rep > 0: continue
                vt = VarianceThreshold(VT_THRESHOLD).fit(XS["S3"][tr]); A, B = vt.transform(XS["S3"][tr]), vt.transform(XS["S3"][te])
                pp, trace = tune_xgb(A, y[tr], CFG["n_trials"], strat, SEED + f, inner_folds(dict(repo=repo[tr], t=raw["t_created"].values[tr], proto="R"), 3, SEED + f))
                pred = make_xgb(**pp).fit(A, y[tr]).predict(B)
                res.append(dict(fold=f, MAE=float(np.mean(np.abs(pred - y[te]))), trace=trace))
            STRATEGY[strat] = res
        json.dump(STRATEGY, open(sp, "w"))
    print({k: np.mean([r["MAE"] for r in v]).round(4) for k, v in STRATEGY.items()})

In [ ]:
# Stage table and paired comparisons for the main claim
STAGE_KEYS = {"A  XGBoost reference": "xgb_ref|S3", "B  + feature selection": "xgb_ref_vt|S3",
              "C  + tuned XGBoost (nested)": "cd_C|S3", "D  + tuned stack (nested)": "cd_D|S3"}
STAGE_TABLE, COMPARISONS = [], []
for lab, key in STAGE_KEYS.items():
    for p in FOLDS:
        s = summarize(key, p); s.update(stage=lab, protocol=p); STAGE_TABLE.append(s)
for p in FOLDS:
    COMPARISONS.append(dict(ref="A", cand="B", **compare("xgb_ref|S3", "xgb_ref_vt|S3", p)))
    COMPARISONS.append(dict(ref="A", cand="C", **compare("xgb_ref|S3", "cd_C|S3", p)))
    COMPARISONS.append(dict(ref="B", cand="C", **compare("xgb_ref_vt|S3", "cd_C|S3", p)))
    COMPARISONS.append(dict(ref="C", cand="D", **compare("cd_C|S3", "cd_D|S3", p)))
    COMPARISONS.append(dict(ref="A", cand="D", **compare("xgb_ref|S3", "cd_D|S3", p)))
    COMPARISONS.append(dict(ref="const", cand="D", **compare("const|S3", "cd_D|S3", p)))
    COMPARISONS.append(dict(ref="xgb_default", cand="C", **compare("xgb_default|S3", "cd_C|S3", p)))
    # leakage ablation: how much accuracy survives when post-hoc and triage fields are removed
    COMPARISONS.append(dict(ref="S0 ref", cand="S3 ref", **compare("xgb_ref|S0", "xgb_ref|S3", p)))
pd.DataFrame(STAGE_TABLE).to_csv(os.path.join(OUT, "table_stages.csv"), index=False)
pd.DataFrame(COMPARISONS).to_csv(os.path.join(OUT, "table_comparisons.csv"), index=False)
print(pd.DataFrame(STAGE_TABLE)[["stage", "protocol", "MAE", "MdAE", "RMSE", "MSE", "R2", "MAE_days", "MdAE_days"]].round(4).to_string(index=False))
print(pd.DataFrame(COMPARISONS)[["ref", "cand", "proto", "mean_diff", "rel_pct", "ci", "p", "test"]].round(4).to_string(index=False))

# stacking weights and selected configurations
EXTRA = [e for e in STORE["cd_C|S3"]["extra"]]
print(pd.DataFrame([{**{"proto": e["proto"], "rep": e["rep"], "fold": e["fold"], "n_selected": e["n_selected"], "alpha": e["alpha"]},
                     **{f"w_{n_}": w for n_, w in zip(["xgb", "rf", "hgb"], e["weights"])}, **{f"xgb_{k}": v for k, v in e["xgb"].items()}} for e in EXTRA]).round(4).to_string(index=False))

## 6. Day-scale analysis, triage ability and feature importance

In [ ]:
def day_bins(d): return pd.cut(d, [-0.5, 0.5, 7.5, 30.5, 365.5, np.inf], labels=["0 d", "1-7 d", "8-30 d", "31-365 d", ">365 d"])
STRAT, TRIAGE = [], []
for lab, key in [("Constant median", "const|S3"), ("A  XGBoost reference", "xgb_ref|S3"), ("C  tuned XGBoost", "cd_C|S3"), ("D  tuned stack", "cd_D|S3")]:
    for p in FOLDS:
        reps = sorted(STORE[key]["oof"][p].keys())
        idx = np.where(~np.isnan(STORE[key]["oof"][p][reps[0]]))[0]
        pr = np.clip(np.expm1(STORE[key]["oof"][p][reps[0]][idx]), 0, None)
        dd = days[idx]; bins = day_bins(dd)
        for b in bins.categories:
            msk = (bins == b)
            STRAT.append(dict(model=lab, protocol=p, bin=str(b), n=int(msk.sum()), MAE_days=float(np.mean(np.abs(pr[msk] - dd[msk]))), bias_days=float(np.mean(pr[msk] - dd[msk]))))
        if lab != "Constant median":
            sc = STORE[key]["oof"][p][reps[0]][idx]
            TRIAGE.append(dict(model=lab, protocol=p, auc_same_day=float(roc_auc_score(dd == 0, -sc)), auc_long_30d=float(roc_auc_score(dd >= 30, sc)),
                               auc_long_90d=float(roc_auc_score(dd >= 90, sc))))
STRAT = pd.DataFrame(STRAT); TRIAGE = pd.DataFrame(TRIAGE)
STRAT.to_csv(os.path.join(OUT, "table_day_scale_strata.csv"), index=False); TRIAGE.to_csv(os.path.join(OUT, "table_triage_auc.csv"), index=False)
print(TRIAGE.round(3).to_string(index=False))

# per-repository error under the grouped protocol (model D, stage table reference)
rep0 = sorted(STORE["cd_D|S3"]["oof"]["G"].keys())[0]
PER_REPO = pd.DataFrame({"repo": repo, "D": np.abs(STORE["cd_D|S3"]["oof"]["G"][rep0] - y), "const": np.abs(STORE["const|S3"]["oof"]["G"][rep0] - y), "n": 1}).groupby("repo").agg(n=("n", "sum"), MAE_D=("D", "mean"), MAE_const=("const", "mean")).sort_values("n", ascending=False)
PER_REPO["improved_over_const"] = PER_REPO.MAE_D < PER_REPO.MAE_const
PER_REPO.to_csv(os.path.join(OUT, "table_per_repository_grouped.csv")); print(PER_REPO.round(3).to_string())

# grouped permutation importance of the reference model on the chronological split (S3)
cols3 = FEATURE_SETS["S3"]
GROUPS = {"Category (4 columns)": CAT_COLS, "Component (105 columns)": COMP_COLS}
for c in ["n_labels"] + ENG_COLS: GROUPS[c] = [c]
_, _, tr, te = FOLDS["T"][0]
mdl = make_xgb(**XGB_REF).fit(XS["S3"][tr], y[tr]); base_mae = np.mean(np.abs(mdl.predict(XS["S3"][te]) - y[te]))
rng = np.random.RandomState(SEED); IMP = []
for g, cs in GROUPS.items():
    ix = [cols3.index(c) for c in cs]; inc = []
    for _ in range(CFG["perm_repeats"]):
        Xp = XS["S3"][te].copy(); perm = rng.permutation(len(te)); Xp[:, ix] = Xp[perm][:, ix]
        inc.append(np.mean(np.abs(mdl.predict(Xp) - y[te])) - base_mae)
    IMP.append(dict(group=g, mae_increase=float(np.mean(inc)), sd=float(np.std(inc))))
IMP = pd.DataFrame(IMP).sort_values("mae_increase", ascending=False); IMP.to_csv(os.path.join(OUT, "table_permutation_importance.csv"), index=False)
print(IMP.round(4).to_string(index=False))

## 7. Same-pipeline replication of the published benchmark

The base paper reports MAE 1.34, MdAE 1.13, MSE 2.83 and RMSE 1.68 for XGBoost, but does not state whether these are computed on the log scale or in days, and does not report hyperparameters. Here the base paper's attribute set (S2: Contributors Expertise, Category, Component) and its 80/20 random hold-out are used with the released data. The metrics are reported on both scales, and a model trained on raw days is added, so the scale of the published figures can be judged.

In [ ]:
BENCH = []
for seed in range(5):
    idx = np.random.RandomState(seed).permutation(n); cutp = int(0.8 * n); tr, te = idx[:cutp], idx[cutp:]
    for lab, build in [("XGBoost reference", b_xgb_ref), ("XGBoost library defaults", b_xgb_default)]:
        m = build().fit(XS["S2"][tr], y[tr]); pl = m.predict(XS["S2"][te]); mm = metrics(y[te], pl, days[te])
        BENCH.append(dict(target="log1p(days)", model=lab, seed=seed, **mm))
        m2 = build().fit(XS["S2"][tr], days[tr]); pdays = np.clip(m2.predict(XS["S2"][te]), 0, None); e = pdays - days[te]
        BENCH.append(dict(target="raw days", model=lab, seed=seed, MAE=float(np.mean(np.abs(e))), MdAE=float(np.median(np.abs(e))), MSE=float(np.mean(e ** 2)),
                          RMSE=float(np.sqrt(np.mean(e ** 2))), R2=float(r2_score(days[te], pdays)), MAE_days=float(np.mean(np.abs(e))), MdAE_days=float(np.median(np.abs(e)))))
BENCH = pd.DataFrame(BENCH)
BENCH_SUM = BENCH.groupby(["target", "model"])[["MAE", "MdAE", "MSE", "RMSE", "R2"]].agg(["mean", "std"]).round(4)
BENCH_SUM.to_csv(os.path.join(OUT, "table_benchmark_replication.csv")); print(BENCH_SUM.to_string())
print("published (base paper): MAE 1.34, MdAE 1.13, MSE 2.83, RMSE 1.68")

## 8. Figures and export

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

# Figure 1: feature-availability ablation under the three protocols
fig, ax = plt.subplots(1, 3, figsize=(10, 3.2), sharey=False)
labs = {"S0": "S0 release set", "S1": "S1 no Comments", "S2": "S2 base paper set", "S3": "S3 planning set", "S3b": "S3 + triage fields", "S3c": "S3 no repo history", "S4": "S4 all features"}
for a, p in zip(ax, ["R", "G", "T"]):
    vals = [summarize(f"xgb_ref|{fs}", p)["MAE"] for fs in labs]; c = summarize("const|S3", p)["MAE"]
    a.bar(range(len(labs)), vals, color="#4C72B0"); a.axhline(c, color="#C44E52", ls="--", lw=1, label="constant median")
    a.set_xticks(range(len(labs))); a.set_xticklabels(list(labs.values()), rotation=40, ha="right"); a.set_title({"R": "Random 5-fold", "G": "Repository-grouped", "T": "Chronological"}[p]); a.set_ylabel("MAE (log scale)")
ax[0].legend(frameon=False); plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig1_feature_set_ablation.png"), dpi=200); plt.close()

# Figure 2: stage comparison under the three protocols
fig, ax = plt.subplots(1, 3, figsize=(10, 3.0))
sl = ["Constant", "A", "B", "C", "D"]; ks = ["const|S3", "xgb_ref|S3", "xgb_ref_vt|S3", "cd_C|S3", "cd_D|S3"]
for a, p in zip(ax, ["R", "G", "T"]):
    a.bar(range(5), [summarize(k, p)["MAE"] for k in ks], color=["#8C8C8C", "#4C72B0", "#55A868", "#DD8452", "#8172B3"]); a.set_xticks(range(5)); a.set_xticklabels(sl)
    a.set_title({"R": "Random 5-fold", "G": "Repository-grouped", "T": "Chronological"}[p]); a.set_ylabel("MAE (log scale)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig2_stage_comparison.png"), dpi=200); plt.close()

# Figure 3: error in days by true duration band
fig, ax = plt.subplots(figsize=(6.5, 3.2)); T_ = STRAT[(STRAT.protocol == "G")]
for i, (lab, g) in enumerate(T_.groupby("model", sort=False)):
    ax.bar(np.arange(len(g)) + i * 0.2, g["MAE_days"].values, width=0.2, label=lab)
ax.set_xticks(np.arange(5) + 0.3); ax.set_xticklabels(["0 d", "1-7 d", "8-30 d", "31-365 d", ">365 d"]); ax.set_yscale("log"); ax.set_ylabel("MAE in days (log axis)"); ax.set_xlabel("True duration band"); ax.legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig3_error_by_duration.png"), dpi=200); plt.close()

# Figure 4: grouped permutation importance
fig, ax = plt.subplots(figsize=(6, 3.6)); I = IMP.head(12)[::-1]
ax.barh(I["group"], I["mae_increase"], xerr=I["sd"], color="#4C72B0"); ax.set_xlabel("Increase in MAE when the feature group is permuted")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig4_permutation_importance.png"), dpi=200); plt.close()

In [ ]:
RESULTS = dict(env=ENV, data=DATA_FACTS, protocols=PROTO_INFO, feature_sets={k: len(v) for k, v in FEATURE_SETS.items()},
               table_E1_E2=T_E12.round(6).to_dict(orient="records"), stages=pd.DataFrame(STAGE_TABLE).round(6).to_dict(orient="records"),
               comparisons=COMPARISONS, strategy=STRATEGY, stacking_and_tuning=[{k: v for k, v in e.items()} for e in EXTRA],
               day_strata=STRAT.round(6).to_dict(orient="records"), triage_auc=TRIAGE.round(6).to_dict(orient="records"),
               per_repository=PER_REPO.reset_index().round(6).to_dict(orient="records"), permutation_importance=IMP.round(6).to_dict(orient="records"),
               benchmark_replication=BENCH.groupby(["target", "model"])[["MAE", "MdAE", "MSE", "RMSE", "R2"]].mean().round(6).reset_index().to_dict(orient="records"),
               sampling_window=SAMPLING.reset_index().astype(str).to_dict(orient="records"), sensitivity_mature=MATURE_TABLE.round(6).to_dict(orient="records"), sensitivity_mature_comparisons=MATURE_CMP,
               runtime_minutes=round((time.time() - t_all) / 60, 1))
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, float) and np.isnan(o): return None
    return o
with open(os.path.join(OUT, "results.json"), "w") as fh: json.dump(_clean(RESULTS), fh, indent=1)
with zipfile.ZipFile("ages_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        if not f.startswith("cache_"): z.write(os.path.join(OUT, f), f)
print("Done. Send ages_outputs.zip (or results.json and the figures) together with the executed notebook.")